# Stock price visualization

Plot prices from `data/prices.csv` for a set of tickers over a date range.

Note: prices are in each listing's **native currency** (USD / KRW / TWD / JPY), so
raw levels aren't comparable across currencies. Use `normalize=True` to rebase each
series to 100 at the start of the range for a fair comparison.

In [1]:
from datetime import date

import polars as pl
import plotly.express as px

PRICES_CSV = "../data/prices.csv"

In [2]:
def load_prices(path=PRICES_CSV):
    """Load the prices file with TS parsed as a date."""
    return pl.read_csv(path, try_parse_dates=True)


def filter_prices(df, tickers, start=None, end=None):
    """Keep rows for `tickers` within the optional [start, end] date range (YYYY-MM-DD)."""
    out = df.filter(pl.col("primary_id").is_in(tickers))
    if start:
        out = out.filter(pl.col("TS") >= date.fromisoformat(start))
    if end:
        out = out.filter(pl.col("TS") <= date.fromisoformat(end))
    return out.sort(["primary_id", "TS"])


def plot_prices(df, tickers, start=None, end=None, normalize=False):
    """Line chart of price per ticker. normalize=True rebases each series to 100."""
    sub = filter_prices(df, tickers, start, end)
    y, ylabel = "value", "Price (native currency)"
    if normalize:
        sub = sub.with_columns(
            (pl.col("value") / pl.col("value").first().over("primary_id") * 100).alias("rebased")
        )
        y, ylabel = "rebased", "Rebased (start = 100)"
    return px.line(
        sub,
        x="TS",
        y=y,
        color="primary_id",
        labels={"TS": "Date", y: ylabel, "primary_id": "Ticker"},
        title="Prices" + (" (rebased to 100)" if normalize else ""),
    )

In [3]:
prices = load_prices()
prices.head()

TS,primary_id,value,currency,TS_RECORDED
date,str,f64,str,"datetime[μs, UTC]"
1972-06-01,"""TXN""",0.687577,"""USD""",2026-06-07 10:51:00.436458 UTC
1972-06-02,"""TXN""",0.700202,"""USD""",2026-06-07 10:51:00.436458 UTC
1972-06-05,"""TXN""",0.701254,"""USD""",2026-06-07 10:51:00.436458 UTC
1972-06-06,"""TXN""",0.671794,"""USD""",2026-06-07 10:51:00.436458 UTC
1972-06-07,"""TXN""",0.669164,"""USD""",2026-06-07 10:51:00.436458 UTC


## Usage
Set the tickers and date range you want, then call `plot_prices`.

In [4]:
# tickers = ["NVDA", "MU", "AMD", "AVGO"]
tickers = ["MU"]
start, end = "1980-01-01", "2024-12-31"

plot_prices(prices, tickers, start, end)

## Compare across currencies
Rebase to 100 so US and foreign listings are comparable.

In [5]:
plot_prices(prices, ["NVDA", "000660.KS", "2408.TW", "285A.T"], start, end, normalize=True)